# 👖 Autoencoders on Fashion MNIST

In this notebook, we'll walk through the steps required to train your own autoencoder on the fashion MNIST dataset.

In [4]:
# ============================================
# Assignment 3 - Autoencoder on Fashion-MNIST
# Full working code for Google Colab
# ============================================

import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models, datasets
import tensorflow.keras.backend as K

# --------------------------------------------
# 1. Load Fashion-MNIST
# --------------------------------------------
(x_train, _), (x_test, _) = datasets.fashion_mnist.load_data()

# Normalize pixel values to [0,1]
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Pad from 28x28 to 32x32 to match textbook-style architecture
x_train = np.pad(x_train, ((0, 0), (2, 2), (2, 2)), mode="constant")
x_test = np.pad(x_test, ((0, 0), (2, 2), (2, 2)), mode="constant")

# Add channel dimension
x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

print("x_train shape:", x_train.shape)
print("x_test shape:", x_test.shape)

# --------------------------------------------
# 2. Set parameters
# --------------------------------------------
IMAGE_SIZE = 32
CHANNELS = 1
EMBEDDING_DIM = 2
BATCH_SIZE = 128
EPOCHS = 5

# --------------------------------------------
# 3. Build Encoder
# --------------------------------------------
encoder_input = layers.Input(
    shape=(IMAGE_SIZE, IMAGE_SIZE, CHANNELS),
    name="encoder_input"
)

x = layers.Conv2D(32, 3, strides=2, padding="same", activation="relu")(encoder_input)
x = layers.Conv2D(64, 3, strides=2, padding="same", activation="relu")(x)
x = layers.Conv2D(128, 3, strides=2, padding="same", activation="relu")(x)

shape_before_flattening = K.int_shape(x)[1:]   # should be (4, 4, 128)
print("Shape before flattening:", shape_before_flattening)

x = layers.Flatten()(x)
encoder_output = layers.Dense(EMBEDDING_DIM, name="encoder_output")(x)

encoder = models.Model(encoder_input, encoder_output, name="encoder")
encoder.summary()

# --------------------------------------------
# 4. Build Decoder
# --------------------------------------------
decoder_input = layers.Input(shape=(EMBEDDING_DIM,), name="decoder_input")

x = layers.Dense(int(np.prod(shape_before_flattening)))(decoder_input)
x = layers.Reshape(shape_before_flattening)(x)

x = layers.Conv2DTranspose(128, 3, strides=2, padding="same", activation="relu")(x)
x = layers.Conv2DTranspose(64, 3, strides=2, padding="same", activation="relu")(x)
x = layers.Conv2DTranspose(32, 3, strides=2, padding="same", activation="relu")(x)
decoder_output = layers.Conv2DTranspose(1, 3, padding="same", activation="sigmoid")(x)

decoder = models.Model(decoder_input, decoder_output, name="decoder")
decoder.summary()

# --------------------------------------------
# 5. Build Autoencoder
# --------------------------------------------
autoencoder = models.Model(
    encoder_input,
    decoder(encoder(encoder_input)),
    name="autoencoder"
)

autoencoder.compile(optimizer="adam", loss="binary_crossentropy")
autoencoder.summary()

# --------------------------------------------
# 6. Train Autoencoder
# --------------------------------------------
history = autoencoder.fit(
    x_train,
    x_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    validation_data=(x_test, x_test)
)

# --------------------------------------------
# 7. Plot training and validation loss
# --------------------------------------------
plt.figure(figsize=(8, 4))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Autoencoder Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

# --------------------------------------------
# 8. Reconstruct 5 test images
# --------------------------------------------
# Change these indices if you want different images
indices = [0, 1, 2, 3, 4]

sample_images = x_test[indices]
reconstructed = autoencoder.predict(sample_images)

plt.figure(figsize=(12, 5))

for i in range(5):
    # Original image
    plt.subplot(2, 5, i + 1)
    plt.imshow(sample_images[i].squeeze(), cmap="gray")
    plt.title(f"Original {i+1}")
    plt.axis("off")

    # Reconstructed image
    plt.subplot(2, 5, i + 6)
    plt.imshow(reconstructed[i].squeeze(), cmap="gray")
    plt.title(f"Recon {i+1}")
    plt.axis("off")

plt.tight_layout()
plt.show()

# --------------------------------------------
# 9. Optional: save models (only if you want)
# --------------------------------------------
# import os
# os.makedirs("models", exist_ok=True)
# autoencoder.save("./models/autoencoder.keras")
# encoder.save("./models/encoder.keras")
# decoder.save("./models/decoder.keras")

2.19.0


## 0. Parameters <a name="parameters"></a>